# What may a faster scheduler change, and what must it leave alone?

The scheduler fixes the analysis before any spectrum is computed. For each frequency $f_j$ it chooses the resolution $r_j$, the segment length $L_j$, the number of averages $K_j$, and the start of every segment. Two implementations build that plan from the same rules. `ltf_plan` walks from $f_j$ to $f_{j+1} = f_j + r_j$ in a loop, applying the LTF compromise between resolution and averaging at every step. `vectorized_ltf_plan`, the default used by `SpectrumAnalyzer`, evaluates the same rules once on a logarithmic grid of $10\,J_{\rm des}$ candidate frequencies and then walks that grid by the same step. Its docstring describes the plan as nearly identical, with differences at the scale of one grid step.

The difference that is allowed is the time it takes. The difference that is not allowed is a plan that breaks $r_j L_j = f_s$ or $r_j = f_{j+1} - f_j$, because those two identities are what make the labeled frequency and the labeled bandwidth the ones the segment actually has. This notebook times the two planners. It does not compare the plans they return; that comparison is the constraint check on $rL$ and on $\Delta f$.

Both functions are called directly, and no spectrum is computed, so no density convention is involved. Four sizes are timed at $f_s = 1000\,\mathrm{Hz}$, overlap 0.5, and `bmin` 1.5. A small plan, $N = 10^5$ with `Jdes` 500, `Lmin` 32, and `Kdes` 50. A typical plan and a high-resolution plan, $N = 10^6$ with `Jdes` 2000 and with `Jdes` $10^4$. A long record, $N = 10^7$ with `Jdes` 2000. The last three use `Lmin` 128 and `Kdes` 100. Each planner is run 10 times and the mean is kept. The speedup is the iterative mean divided by the vectorized mean.

In [ ]:
import timeit
import numpy as np
import pandas as pd
import math
import sys
import logging
import matplotlib.pyplot as plt
from IPython.display import display, HTML  # safe table display without .style

from speckit.schedulers import vectorized_ltf_plan, ltf_plan

In [ ]:
test_scenarios = [
    {"description": "Small", "params": {'N': 100_000, 'Jdes': 500, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 32, 'Kdes': 50}},
    {"description": "Typical", "params": {'N': 1_000_000, 'Jdes': 2000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
    {"description": "High-Res (Jdes)", "params": {'N': 1_000_000, 'Jdes': 10000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
    {"description": "Long Data (N)", "params": {'N': 10_000_000, 'Jdes': 2000, 'fs': 1000, 'olap': 0.5, 'bmin': 1.5, 'Lmin': 128, 'Kdes': 100}},
]
num_runs = 10
results_data = []

print("Comparing Performance of LTF Schedulers (running...)")

vectorized_ltf_plan(**(test_scenarios[0]["params"]))

for scenario in test_scenarios:
    params = scenario["params"]
    
    # Use globals() in notebook environment for timeit
    iter_times = timeit.repeat(lambda: ltf_plan(**params), number=1, repeat=num_runs)
    vec_times = timeit.repeat(lambda: vectorized_ltf_plan(**params), number=1, repeat=num_runs)

    iter_avg_time = np.mean(iter_times)
    vec_avg_time = np.mean(vec_times)
    speedup = iter_avg_time / vec_avg_time if vec_avg_time > 0 else float('inf')
    
    results_data.append({
        "Scenario": scenario["description"],
        "Iterative Time (s)": iter_avg_time,
        "Vectorized Time (s)": vec_avg_time,
        "Speedup (x)": speedup
    })

In [ ]:
results_df = pd.DataFrame(results_data)

# Make a formatted copy for display
fmt_df = results_df.copy()
fmt_df["Iterative Time (s)"] = fmt_df["Iterative Time (s)"].map(lambda v: f"{v:.4f}")
fmt_df["Vectorized Time (s)"] = fmt_df["Vectorized Time (s)"].map(lambda v: f"{v:.4f}")
fmt_df["Speedup (x)"] = fmt_df["Speedup (x)"].map(lambda v: f"{v:.2f}x")

caption_html = "<h3>Performance Benchmark Results</h3>"
table_html = fmt_df.to_html(index=False, escape=False)  # pure pandas HTML, no jinja2
display(HTML(caption_html + table_html))

In [ ]:
labels = results_df['Scenario']
iter_times = results_df['Iterative Time (s)']
vec_times = results_df['Vectorized Time (s)']
speedups = results_df['Speedup (x)']
x = np.arange(len(labels))
width = 0.35

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)
fig.suptitle('LTF Scheduler Performance Comparison', fontsize=16)

# --- Plot 1: Execution Time ---
rects1 = ax1.bar(x - width/2, iter_times, width, label='Iterative (ltf_plan)', color='skyblue')
rects2 = ax1.bar(x + width/2, vec_times, width, label='Vectorized', color='coral')
ax1.set_ylabel('Execution Time (seconds)')
ax1.set_title('Execution Time per Scenario (Log Scale)')
ax1.set_yscale('log')
ax1.legend()
ax1.bar_label(rects1, padding=3, fmt='%.3f')
ax1.bar_label(rects2, padding=3, fmt='%.3f')
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# --- Plot 2: Speedup Factor ---
rects3 = ax2.bar(x, speedups, color='mediumseagreen')
ax2.set_ylabel('Speedup Factor (x)')
ax2.set_title('Speedup Factor of Vectorized vs. Iterative')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=0)
ax2.bar_label(rects3, padding=3, fmt='%.1fx')
ax2.axhline(1, color='grey', linestyle='--', lw=1) # Add a line at 1x for reference
ax2.grid(axis='y', linestyle='--', alpha=0.7)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## Reading the result

The table gives the mean time of each planner and the speedup for each plan size. The figure shows the same numbers: paired bars for the two times, on a logarithmic scale, and the speedup against a dashed line at $1\times$.

Read the speedup against 1, and read how it moves from one plan to the next. From the typical plan to the high-resolution plan, `Jdes` grows at fixed $N$: the iterative loop pays one step per frequency, and the vectorized planner pays a larger candidate grid. From the typical plan to the long record, $N$ grows at fixed `Jdes`, and both planners pay for building the segment starts. No expected speedup is stated; the number belongs to the machine the cell runs on.

The vectorized planner is called once before the timings, so its first plan size is not carrying an initialization the iterative planner still might. Because its walk reads $r$, $L$, and $K$ from the first grid point at or above the current frequency, the plan can differ from `ltf_plan` by as much as one grid step. That difference is not measured here.